# PyTorch Autoencoder Unsupervised Learning

This notebook learns compact latent vectors for hourly weather states using CPU-friendly dense autoencoders. The benchmark compares several bottleneck sizes side by side and inspects unusual high-error hours in the held-out time slice.

The workflow is:

- build a bounded multivariate hourly weather matrix from the canonical table
- train several dense autoencoders side by side
- compare reconstruction error and latent-space structure
- inspect PCA projections and high-error hours

In [ ]:
from __future__ import annotations

import os
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device('cpu')
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print({'device': str(device), 'torch_version': torch.__version__})

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {name: np.asarray(values, dtype=float) for name, values in table.columns.items()}
timestamps = np.asarray(table.timestamps)

feature_names = [
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__pressure_msl',
    'misc__precipitation',
    'wind__wind_speed_10m',
    'wind__wind_gusts_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'european_air_quality_index__european_aqi',
    'additional_vars_and_options__is_day',
]
missing = [name for name in feature_names if name not in column_arrays]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

X = np.column_stack([column_arrays[name] for name in feature_names]).astype(np.float32)
valid_mask = np.isfinite(X).all(axis=1)
X = X[valid_mask]
time_index = timestamps[valid_mask]
max_rows = 32000
if len(X) > max_rows:
    X = X[-max_rows:]
    time_index = time_index[-max_rows:]

print(f'Rows: {len(X):,}')
print(f'Feature count: {X.shape[1]}')
print(f'Subset start: {time_index[0]}')
print(f'Subset end:   {time_index[-1]}')

In [ ]:
def chronological_split(values: np.ndarray, time_values: np.ndarray, train_ratio: float = 0.7, valid_ratio: float = 0.15):
    total = len(values)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        'X_train': values[:train_end],
        't_train': time_values[:train_end],
        'X_valid': values[train_end:valid_end],
        't_valid': time_values[train_end:valid_end],
        'X_test': values[valid_end:],
        't_test': time_values[valid_end:],
    }
split = chronological_split(X, time_index)
x_mean = split['X_train'].mean(axis=0)
x_std = split['X_train'].std(axis=0) + 1e-6
def normalize_features(values: np.ndarray) -> np.ndarray:
    return ((values - x_mean) / x_std).astype(np.float32)
def to_tensor_dataset(values: np.ndarray) -> TensorDataset:
    return TensorDataset(torch.from_numpy(normalize_features(values)))
train_loader = DataLoader(to_tensor_dataset(split['X_train']), batch_size=256, shuffle=True, num_workers=0)
valid_loader = DataLoader(to_tensor_dataset(split['X_valid']), batch_size=512, shuffle=False, num_workers=0)
test_loader = DataLoader(to_tensor_dataset(split['X_test']), batch_size=512, shuffle=False, num_workers=0)
for name in ['train', 'valid', 'test']:
    print(name, split[f'X_{name}'].shape, split[f't_{name}'][0], split[f't_{name}'][-1])

In [ ]:
class DenseAutoencoder(nn.Module):
    def __init__(self, input_size: int, hidden_layers: list[int], latent_dim: int, dropout: float = 0.0):
        super().__init__()
        encoder_layers = []
        prev_size = input_size
        for hidden_size in hidden_layers:
            encoder_layers.append(nn.Linear(prev_size, hidden_size))
            encoder_layers.append(nn.ReLU())
            if dropout > 0:
                encoder_layers.append(nn.Dropout(dropout))
            prev_size = hidden_size
        encoder_layers.append(nn.Linear(prev_size, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        decoder_layers = []
        prev_size = latent_dim
        for hidden_size in reversed(hidden_layers):
            decoder_layers.append(nn.Linear(prev_size, hidden_size))
            decoder_layers.append(nn.ReLU())
            prev_size = hidden_size
        decoder_layers.append(nn.Linear(prev_size, input_size))
        self.decoder = nn.Sequential(*decoder_layers)
    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        latent = self.encoder(inputs)
        reconstruction = self.decoder(latent)
        return reconstruction, latent
def reconstruction_errors(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    errors = []
    with torch.no_grad():
        for (features,) in loader:
            reconstruction, _ = model(features)
            batch_errors = torch.mean((reconstruction - features) ** 2, dim=1)
            errors.append(batch_errors.detach().numpy())
    return np.concatenate(errors)
def collect_latents(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    latents = []
    with torch.no_grad():
        for (features,) in loader:
            _, latent = model(features)
            latents.append(latent.detach().numpy())
    return np.concatenate(latents)
def train_model(model: nn.Module, train_data: DataLoader, valid_data: DataLoader, epochs: int = 12):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    history = []
    best_valid_loss = float('inf')
    best_state = None
    for epoch in range(1, epochs + 1):
        model.train()
        batch_losses = []
        for (features,) in train_data:
            optimizer.zero_grad()
            reconstruction, _ = model(features)
            loss = loss_fn(reconstruction, features)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            batch_losses.append(loss.item())
        train_loss = float(np.mean(batch_losses))
        valid_loss = float(np.mean(reconstruction_errors(model, valid_data)))
        history.append({'epoch': epoch, 'train_loss': train_loss, 'valid_loss': valid_loss})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = {name: tensor.detach().clone() for name, tensor in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

In [ ]:
benchmark_specs = {
    'autoencoder_shallow': {'hidden_layers': [32], 'latent_dim': 8, 'dropout': 0.0},
    'autoencoder_medium': {'hidden_layers': [64, 32], 'latent_dim': 8, 'dropout': 0.05},
    'autoencoder_bottleneck': {'hidden_layers': [64, 48], 'latent_dim': 4, 'dropout': 0.05},
}
benchmark_rows = []
histories = {}
latent_results = {}
for model_name, spec in benchmark_specs.items():
    model = DenseAutoencoder(input_size=X.shape[1], hidden_layers=spec['hidden_layers'], latent_dim=spec['latent_dim'], dropout=spec['dropout'])
    history = train_model(model, train_loader, valid_loader)
    valid_errors = reconstruction_errors(model, valid_loader)
    test_errors = reconstruction_errors(model, test_loader)
    latents = collect_latents(model, test_loader)
    reduced = PCA(n_components=2, random_state=42).fit_transform(latents)
    clusters = KMeans(n_clusters=4, n_init=20, random_state=42).fit_predict(latents)
    benchmark_rows.append({'model': model_name, 'valid_error_mean': float(np.mean(valid_errors)), 'test_error_mean': float(np.mean(test_errors)), 'latent_std_mean': float(np.mean(latents.std(axis=0)))})
    histories[model_name] = history
    latent_results[model_name] = {'test_errors': test_errors, 'reduced': reduced, 'clusters': clusters}
for row in sorted(benchmark_rows, key=lambda item: item['test_error_mean']):
    print(row)

In [ ]:
best_model_name = min(benchmark_rows, key=lambda item: item['test_error_mean'])['model']
best_result = latent_results[best_model_name]
fig, axes = plt.subplots(1, len(histories), figsize=(5 * len(histories), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (model_name, history) in zip(axes, histories.items()):
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train_loss'] for row in history]
    valid_loss = [row['valid_loss'] for row in history]
    ax.plot(epochs, train_loss, label='train')
    ax.plot(epochs, valid_loss, label='valid')
    ax.set_title(model_name)
    ax.set_xlabel('Epoch')
    ax.legend()
axes[0].set_ylabel('Reconstruction MSE')
plt.tight_layout()
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(best_result['reduced'][:, 0], best_result['reduced'][:, 1], c=best_result['clusters'], cmap='tab10', s=18, alpha=0.75)
ax.set_title(f'PCA view of latent space for {best_model_name}')
ax.set_xlabel('PCA 1')
ax.set_ylabel('PCA 2')
plt.colorbar(scatter, ax=ax, label='KMeans cluster')
plt.tight_layout()
top_order = np.argsort(best_result['test_errors'])[-12:][::-1]
fig, ax = plt.subplots()
ax.plot(split['t_test'], best_result['test_errors'], label='reconstruction error', linewidth=1.4)
ax.scatter(split['t_test'][top_order], best_result['test_errors'][top_order], color='crimson', label='top high-error hours', zorder=3)
ax.set_title(f'High-error hours for {best_model_name}')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Reconstruction error')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
print('Top high-error test hours for best model:')
for index in top_order:
    print(split['t_test'][index], float(best_result['test_errors'][index]))